# Amazon Product Reviews — Visualizations

Using the pre-cleaned dataset (Amazon_Product_Review_Cleaned.csv).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif'})

In [ ]:
df = pd.read_csv('../Datasets/Amazon_Product_Review_Cleaned.csv')
print(f'Shape: {df.shape}')
print(f'Sentiment counts:\n{df["sentiment"].value_counts().to_string()}')
df[['name', 'device_type', 'reviews.rating', 'sentiment', 'price_usd']].head()

---
## 1 — Rating Distribution (Matplotlib Histogram)

In [ ]:
star_colors = {1: '#d32f2f', 2: '#f57c00', 3: '#fbc02d', 4: '#388e3c', 5: '#1565c0'}
rating_counts = df['reviews.rating'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    [f'{int(r)} star' for r in rating_counts.index],
    rating_counts.values,
    color=[star_colors[int(r)] for r in rating_counts.index],
    width=0.55, edgecolor='white', linewidth=1.2
)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 8, f'{int(h):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Rating Distribution', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_ylim(0, rating_counts.max() * 1.15)
ax.spines[['top', 'right']].set_visible(False)
patches = [mpatches.Patch(color=star_colors[i], label=f'{i} Star') for i in sorted(star_colors)]
ax.legend(handles=patches, loc='upper left', framealpha=0.6, fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/charts/hist_rating_distribution.png', dpi=150)
plt.show()

---
## 2 — Sentiment Breakdown (Plotly Interactive Donut)

In [ ]:
sent_counts = df['sentiment'].value_counts().reset_index()
sent_counts.columns = ['Sentiment', 'Count']
color_map = {'Positive': '#43a047', 'Neutral': '#fdd835', 'Negative': '#e53935'}

fig = px.pie(
    sent_counts, names='Sentiment', values='Count',
    hole=0.45, color='Sentiment', color_discrete_map=color_map,
    title='Sentiment Breakdown'
)
fig.update_traces(
    textposition='inside', textinfo='percent+label',
    hovertemplate='%{label}: %{value} reviews (%{percent})'
)
fig.update_layout(title_font_size=18, showlegend=True,
                  legend=dict(orientation='v', x=1.02),
                  margin=dict(t=60, b=20, l=20, r=20))
fig.show()

---
## 3 — Top 15 Most Reviewed Products (Plotly Interactive Bar)

In [ ]:
top15 = (
    df.groupby('name')
      .agg(review_count=('reviews.text', 'count'), avg_rating=('reviews.rating', 'mean'))
      .sort_values('review_count', ascending=False)
      .head(15).reset_index()
)
top15['avg_rating'] = top15['avg_rating'].round(2)

fig = px.bar(
    top15.sort_values('review_count'), x='review_count', y='name',
    orientation='h', color='avg_rating',
    color_continuous_scale='RdYlGn', range_color=[1, 5],
    text='review_count',
    labels={'review_count': 'Number of Reviews', 'name': '', 'avg_rating': 'Avg Rating'},
    title='Top 15 Most Reviewed Products'
)
fig.update_traces(textposition='outside',
                  hovertemplate='<b>%{y}</b><br>Reviews: %{x}<br>Avg Rating: %{marker.color:.2f}')
fig.update_layout(title_font_size=18, height=520,
                  coloraxis_colorbar=dict(title='Avg Rating'),
                  margin=dict(t=60, b=40, l=20, r=20))
fig.show()

---
## 4 — Correlation Heatmap (Seaborn)

In [ ]:
df['sentiment_score'] = df['sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})
df['reviews.numHelpful'] = df['reviews.numHelpful'].fillna(0)

corr_df = df[['reviews.rating', 'reviews.numHelpful', 'sentiment_score', 'price_usd']].rename(columns={
    'reviews.rating': 'Rating', 'reviews.numHelpful': 'Helpful Votes',
    'sentiment_score': 'Sentiment', 'price_usd': 'Price (USD)'
})
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, linewidths=1.5, linecolor='white',
            square=True, annot_kws={'size': 12, 'weight': 'bold'}, ax=ax)
ax.set_title('Review Correlation Heatmap', fontsize=14, fontweight='bold', pad=14)
ax.tick_params(axis='x', labelsize=10, rotation=20)
ax.tick_params(axis='y', labelsize=10, rotation=0)
plt.tight_layout()
plt.savefig('../outputs/charts/heatmap_correlations.png', dpi=150)
plt.show()

---
## 5 — Sentiment by Device Type (Plotly Interactive)

In [ ]:
device_sent = (
    df.groupby(['device_type', 'sentiment'])
      .size().reset_index(name='count')
)
fig = px.bar(
    device_sent, x='device_type', y='count', color='sentiment',
    barmode='group',
    color_discrete_map={'Positive': '#43a047', 'Neutral': '#fdd835', 'Negative': '#e53935'},
    labels={'device_type': 'Device', 'count': 'Reviews', 'sentiment': 'Sentiment'},
    title='Sentiment Breakdown by Device Type'
)
fig.update_layout(title_font_size=18, xaxis_title='Device Type',
                  yaxis_title='Number of Reviews', legend_title='Sentiment',
                  margin=dict(t=60, b=40))
fig.show()

---
## 6 — Price Distribution by Device Type (Matplotlib Boxplot)

In [ ]:
price_data = df.dropna(subset=['price_usd'])
device_order = price_data.groupby('device_type')['price_usd'].median().sort_values().index.tolist()
groups = [price_data[price_data['device_type'] == d]['price_usd'].values for d in device_order]

fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(groups, labels=device_order, patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                boxprops=dict(linewidth=1.2), whiskerprops=dict(linewidth=1.2),
                flierprops=dict(marker='o', markersize=4, alpha=0.4))
for patch, color in zip(bp['boxes'], plt.cm.Set2.colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_title('Price Distribution by Device Type', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Device Type', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/boxplot_price_by_device.png', dpi=150)
plt.show()

---
## 7 — Stacked Sentiment: Top 10 Products (Matplotlib)

In [ ]:
top10_names = top15['name'].head(10).tolist()
stacked = (
    df[df['name'].isin(top10_names)]
    .groupby(['name', 'sentiment'])
    .size().unstack(fill_value=0)
    .reindex(columns=['Positive', 'Neutral', 'Negative'], fill_value=0)
)
stacked = stacked.loc[top10_names]
stacked.index = [n[:35] + '...' if len(n) > 35 else n for n in stacked.index]

fig, ax = plt.subplots(figsize=(12, 6))
stacked.plot(kind='barh', stacked=True,
             color=['#43a047', '#fdd835', '#e53935'], edgecolor='white', ax=ax)
ax.set_title('Sentiment Breakdown - Top 10 Products', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_ylabel('')
ax.legend(title='Sentiment', loc='lower right', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/stacked_sentiment_top10.png', dpi=150)
plt.show()

---
## 8 — Products with Inconsistent Ratings (Matplotlib)

In [ ]:
inconsistent = (
    df.groupby('name')['reviews.rating']
      .agg(avg_rating='mean', std_rating='std', review_count='count')
      .dropna(subset=['std_rating'])
      .query('review_count >= 10')
      .sort_values('std_rating', ascending=False)
      .head(15).reset_index()
)
inconsistent['avg_rating'] = inconsistent['avg_rating'].round(2)
inconsistent['std_rating'] = inconsistent['std_rating'].round(2)
inconsistent['short_name'] = inconsistent['name'].apply(
    lambda n: n[:30] + '...' if len(n) > 30 else n
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    inconsistent['short_name'][::-1],
    inconsistent['std_rating'][::-1],
    color='#ef6c00', edgecolor='white', height=0.6, alpha=0.85
)
for bar, (_, row) in zip(bars, inconsistent[::-1].iterrows()):
    w = bar.get_width()
    ax.text(w + 0.01, bar.get_y() + bar.get_height() / 2,
            f'std={row["std_rating"]}  avg={row["avg_rating"]}  n={int(row["review_count"])}',
            va='center', fontsize=8.5)
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=1.2, alpha=0.6, label='std=1.0 threshold')
ax.set_title('Top 15 Products with Most Inconsistent Ratings', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Rating Std Deviation  (higher = more inconsistent)', fontsize=11)
ax.set_xlim(0, inconsistent['std_rating'].max() * 1.35)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/inconsistent_ratings.png', dpi=150)
plt.show()

print(inconsistent[['name', 'avg_rating', 'std_rating', 'review_count']].to_string(index=False))